# Contrastive Experiments Analysis

This notebook analyzes the performance of different contrastive experiments using multiple models (Qwen3-14B, Qwen3-32B). We compare different non-activating sources (random, co-occurrence, faiss, decoder_similarity) and ranking strategies (top vs quantiles) across multiple metrics.

## Analysis Overview

- **Mean Frequency-Weighted F1 Scores**: Bar charts comparing performance across configurations
- **Performance Distribution**: KDE plots showing density distributions of F1 scores
- **Comprehensive Comparison**: Side-by-side analysis of all available contrastive experiments

## Configuration Pattern

Experiments follow the pattern: `{model}_{source}_{mode}_{ranking}`
- **Sources**: random (baseline), co-occurrence, faiss, decoder_similarity
- **Modes**: contrastive_scorer_only, baseline
- **Ranking**: top, quantiles

## 1. Setup and Configuration

Import required libraries and set up the analysis environment.

In [ ]:
import sys
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from scipy import stats
from tqdm.auto import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from bootstrap_ci import compute_weighted_ci_errors

# Add the parent directory to the path to import delphi modules
sys.path.append(str(Path.cwd().parent))

from delphi.log.result_analysis import (
    import_plotly,
    load_data,
    get_agg_metrics,
    add_latent_f1,
    compute_confusion,
    compute_classification_metrics,
    frequency_weighted_f1
)

# Import plotly for plotting
px = import_plotly()

# Configuration
EXPERIMENT_DIR = "Contrastive Scoring Quick"
CONFIDENCE_LEVEL = 0.95
ENABLE_BOOTSTRAP = True
BOOTSTRAP_SAMPLES = 1000
N_JOBS = 8  # Number of threads for parallel bootstrap processing

# Set up directories
results_dir = Path.cwd().parent / "results"
visualizations_dir = results_dir / "visualizations" / "contrastive_experiments"
visualizations_dir.mkdir(exist_ok=True, parents=True)

print(f"Experiment directory: {EXPERIMENT_DIR}")
print(f"Results directory: {results_dir}")
print(f"Visualizations output: {visualizations_dir}")

# Define color scheme for different experiment types
SOURCE_COLORS = {
    'random': '#E53E3E',        # Red for baseline
    'co-occurrence': '#3182CE', # Blue
    'faiss': '#38A169',         # Green  
    'decoder_similarity': '#805AD5'  # Purple
}

RANKING_STYLES = {
    'top': 'solid',
    'quantiles': 'dashed'
}

# Clean display name mappings
SOURCE_DISPLAY = {
    'random': 'Random',
    'co-occurrence': 'Co-occurrence',
    'faiss': 'FAISS',
    'decoder_similarity': 'Decoder Sim.'
}

RANKING_DISPLAY = {
    'top': 'Top',
    'quantiles': 'Quantiles'
}

def parse_experiment_name(exp_name):
    """Parse experiment directory name to extract components."""
    # Remove pythiaST_ prefix
    name = exp_name.replace('pythiaST_', '')
    
    # Extract model name (first part before the source)
    # Pattern: Qwen3_14B_quantized_w4a16 or Qwen3_32B_quantized_w4a16
    model = None
    model_display = None
    if 'Qwen3_14B' in name:
        model = 'Qwen3_14B'
        model_display = 'Qwen3-14B'
    elif 'Qwen3_32B' in name:
        model = 'Qwen3_32B'
        model_display = 'Qwen3-32B'
    else:
        # Try to extract the first part as model name
        parts = name.split('_')
        if len(parts) > 1:
            model = '_'.join(parts[:2])  # Take first two parts as model name
            model_display = model
    
    # Find the source (random, co-occurrence, faiss, decoder)
    source = None
    if 'random' in name:
        source = 'random'
    elif 'co-occurrence' in name:
        source = 'co-occurrence'
    elif 'faiss' in name:
        source = 'faiss'
    elif 'decoder' in name:
        source = 'decoder_similarity'
    
    # Find the ranking strategy
    ranking = 'quantiles' if 'quantiles' in name else 'top'
    
    # Determine mode
    mode = 'baseline' if 'baseline' in name else 'contrastive'
    
    # Create clean display name
    source_display = SOURCE_DISPLAY.get(source, source)
    ranking_display = RANKING_DISPLAY.get(ranking, ranking)
    display_name = f"{source_display} ({ranking_display})"
    
    return {
        'model': model,
        'model_display': model_display,
        'source': source,
        'ranking': ranking,
        'mode': mode,
        'display_name': display_name
    }

# Check available experiments
experiments_base = results_dir / "pythiaST" / EXPERIMENT_DIR
if experiments_base.exists():
    print(f"\nAvailable contrastive experiments:")
    model_groups = {}
    for exp_dir in sorted(experiments_base.glob('pythiaST_*')):
        config = parse_experiment_name(exp_dir.name)
        model_name = config['model_display']
        if model_name not in model_groups:
            model_groups[model_name] = []
        model_groups[model_name].append(config['display_name'])
        print(f"  - {exp_dir.name} -> {model_name}: {config['display_name']}")
    
    print(f"\nFound {len(model_groups)} model types:")
    for model_name, experiments in model_groups.items():
        print(f"  {model_name}: {len(experiments)} experiments")
else:
    print(f"Warning: Experiments directory not found: {experiments_base}")

## 2. Load Contrastive Experiment Results

Load results from all available contrastive experiments and process them for analysis.

In [ ]:
def load_contrastive_results(results_dir: Path, experiment_dir: str):
    """Load results from all contrastive experiments."""
    experiment_results = {}
    experiment_stats = {}
    
    experiments_base = results_dir / "pythiaST" / experiment_dir
    
    if not experiments_base.exists():
        print(f"Warning: Experiments directory not found: {experiments_base}")
        return experiment_results, experiment_stats
    
    for exp_dir in experiments_base.glob('pythiaST_*'):
        config = parse_experiment_name(exp_dir.name)
        # Create unique key that includes model name
        unique_key = f"{config['model_display']}: {config['display_name']}"
        
        scores_path = exp_dir / "scores"
        if scores_path.exists():
            try:
                # Use the shared cache from the parent pythiaST directory
                latents_path = results_dir / "pythiaST" / "cache" / "latents"
                if not latents_path.exists():
                    latents_path = exp_dir / "latents"  # Fallback to local latents
                
                if latents_path.exists():
                    # Extract module names from the actual files
                    sample_score_dir = next(scores_path.iterdir())
                    sample_files = list(sample_score_dir.glob("*.txt"))
                    if sample_files:
                        # Extract module name from filename pattern
                        sample_filename = sample_files[0].stem
                        module_name = sample_filename.split('_latent')[0]
                        modules = [module_name]
                    else:
                        print(f"No score files found in {sample_score_dir}")
                        continue
                    
                    # Use load_data from result_analysis.py
                    latent_df, counts = load_data(scores_path, latents_path, modules)
                    
                    if latent_df.empty:
                        print(f"No data found for {unique_key}")
                        continue
                    
                    # Use add_latent_f1 and get_agg_metrics from result_analysis.py
                    latent_df = add_latent_f1(latent_df)
                    processed_df = get_agg_metrics(latent_df, counts)
                    
                    experiment_results[unique_key] = {
                        'latent_df': latent_df,
                        'processed_df': processed_df,
                        'counts': counts,
                        'config': config
                    }
                else:
                    print(f"Latents path not found for {display_name}")
            
            except Exception as e:
                print(f"Error loading results for {exp_dir.name}: {e}")
                continue
        
        # Load experiment statistics
        stats_file = exp_dir / "explainer_stats.json"
        if stats_file.exists():
            try:
                with open(stats_file, 'r') as f:
                    stats = json.load(f)
                    experiment_stats[unique_key] = stats
            except Exception as e:
                print(f"Error loading stats for {exp_dir.name}: {e}")
                experiment_stats[unique_key] = None
        else:
            experiment_stats[unique_key] = None
    
    return experiment_results, experiment_stats

# Load all contrastive experiment results
print("Loading contrastive experiment results...")
experiment_results, experiment_stats = load_contrastive_results(results_dir, EXPERIMENT_DIR)

print(f"\nLoaded results for {len(experiment_results)} experiments:")
for exp_name in experiment_results.keys():
    print(f"  - {exp_name}")

# Display sample metrics for the first experiment
if experiment_results:
    sample_exp = list(experiment_results.keys())[0]
    sample_data = experiment_results[sample_exp]['processed_df']
    print(f"\nSample metrics from {sample_exp}:")
    print(sample_data[['score_type', 'accuracy', 'f1_score', 'precision', 'recall', 'weighted_f1']].round(3))

## 3. Generate Mean Frequency-Weighted F1 Bar Charts

Create bar charts showing mean frequency-weighted F1 scores across all contrastive experiments, organized by source and ranking strategy.

In [ ]:
# Bootstrap CI computation is now imported from bootstrap_ci module

def compute_mean_f1_across_score_types(experiment_results, score_types=['detection', 'fuzz'], enable_bootstrap=ENABLE_BOOTSTRAP, n_boot=BOOTSTRAP_SAMPLES, confidence=CONFIDENCE_LEVEL, n_jobs=N_JOBS):
    """Compute mean frequency-weighted F1 across multiple score types (detection and fuzz) and CI errors."""
    rows = []
    
    for exp_name, data in tqdm(experiment_results.items(), desc="Computing mean F1 with bootstrap CIs"):
        processed_df = data['processed_df']
        latent_df = data['latent_df']
        counts = data['counts']
        config = data['config']
        
        # Collect F1 scores and error bars for each score type
        f1_scores = []
        lower_errors = []
        upper_errors = []
        
        for score_type in score_types:
            score_row = processed_df[processed_df['score_type'] == score_type]
            if len(score_row) > 0 and 'weighted_f1' in score_row.columns:
                freq_weighted_f1 = score_row['weighted_f1'].iloc[0]
                if freq_weighted_f1 is not None:
                    f1_scores.append(float(freq_weighted_f1))
                    
                    # Compute bootstrap CI for this score type using the optimized function
                    if enable_bootstrap and counts is not None:
                        score_subset = latent_df[latent_df['score_type'] == score_type]
                        if len(score_subset) > 0:
                            try:
                                lower_err, upper_err = compute_weighted_ci_errors(
                                    score_subset, counts, freq_weighted_f1, 
                                    confidence, n_boot, n_jobs, use_cache=True
                                )
                                lower_errors.append(lower_err)
                                upper_errors.append(upper_err)
                            except Exception as e:
                                print(f"Warning: Bootstrap CI failed for {exp_name} {score_type}: {e}")
                                lower_errors.append(0.0)
                                upper_errors.append(0.0)
                        else:
                            lower_errors.append(0.0)
                            upper_errors.append(0.0)
                    else:
                        lower_errors.append(0.0)
                        upper_errors.append(0.0)
        
        # Calculate mean F1 across available score types
        if len(f1_scores) == 0:
            rows.append({
                'experiment': config['display_name'],
                'model': config['model_display'],
                'source': config['source'],
                'ranking': config['ranking'],
                'mode': config['mode'],
                'frequency_weighted_f1': None,
                'ci_lower_error': 0.0,
                'ci_upper_error': 0.0,
                'n_score_types': 0
            })
            continue
        
        mean_f1 = np.mean(f1_scores)
        
        # Average the error bars across score types
        # This is a conservative approximation for the CI of the mean
        if len(lower_errors) > 0:
            mean_lower_error = np.mean(lower_errors)
            mean_upper_error = np.mean(upper_errors)
        else:
            mean_lower_error = 0.0
            mean_upper_error = 0.0
        
        rows.append({
            'experiment': config['display_name'],
            'model': config['model_display'],
            'source': config['source'],
            'ranking': config['ranking'],
            'mode': config['mode'],
            'frequency_weighted_f1': float(mean_f1),
            'ci_lower_error': mean_lower_error,
            'ci_upper_error': mean_upper_error,
            'n_score_types': len(f1_scores)
        })
    
    return pd.DataFrame(rows)

# Generate bar charts using mean of detection and fuzz score types
all_score_types = set()
for data in experiment_results.values():
    all_score_types.update(list(data['processed_df']['score_type'].unique()))
all_score_types = sorted(list(all_score_types))

print(f"Available score types: {all_score_types}")
print(f"Computing mean F1 across detection and fuzz (bootstrap={ENABLE_BOOTSTRAP})")

# Compute mean F1 and error bars across detection and fuzz
mean_error_table = compute_mean_f1_across_score_types(experiment_results, score_types=['detection', 'fuzz'])

# Create bar charts - separate plots for each model (using mean of detection and fuzz)
if mean_error_table.empty:
    print("No data available for bar charts")
else:
    # Get unique models
    models = mean_error_table['model'].unique()
    
    # Create a separate plot for each model
    for model_name in sorted(models):
        model_df = mean_error_table[mean_error_table['model'] == model_name].copy()
        
        if model_df.empty:
            continue
        
        # Calculate random baseline F1 from class distribution (mean across detection and fuzz)
        # Get the first experiment result for this model to compute baseline
        model_experiments = [k for k, v in experiment_results.items() if v['config']['model_display'] == model_name]
        if model_experiments:
            first_exp = experiment_results[model_experiments[0]]
            baseline_f1s = []
            for score_type in ['detection', 'fuzz']:
                score_subset = first_exp['latent_df'][first_exp['latent_df']['score_type'] == score_type]
                if len(score_subset) > 0:
                    # Calculate overall positive rate across all examples
                    total_positives = score_subset['activating'].sum()
                    total_examples = len(score_subset)
                    p = total_positives / total_examples if total_examples > 0 else 0.5
                    
                    # For a 50/50 random predictor on imbalanced data:
                    # Precision = P(true|pred=1) = p/2 / (p/2 + (1-p)/2) = p
                    # Recall = P(pred=1|true) = 0.5
                    # F1 = 2pr/(p+r) = 2(p)(0.5)/(p+0.5) = p/(p+0.5)
                    random_f1 = p / (p + 0.5) if (p + 0.5) > 0 else 0
                    baseline_f1s.append(random_f1)
            
            random_baseline_f1 = np.mean(baseline_f1s) if len(baseline_f1s) > 0 else 0.5
        else:
            random_baseline_f1 = 0.5
        
        # Sort by frequency_weighted_f1 for better visualization
        model_df = model_df.sort_values('frequency_weighted_f1', ascending=False)
        
        # Create colors based on source
        colors = [SOURCE_COLORS.get(source, '#808080') for source in model_df['source']]
        
        fig = px.bar(
            model_df,
            x='experiment',
            y='frequency_weighted_f1',
            color='source',
            color_discrete_map=SOURCE_COLORS,
            title=f'{model_name} - Frequency-Weighted F1 Score - Mean (Detection + Fuzz) ({int(CONFIDENCE_LEVEL*100)}% CI)',
            text='frequency_weighted_f1',
            hover_data=['ranking', 'mode', 'n_score_types']
        )
        
        # Add error bars
        fig.update_traces(
            error_y=dict(
                type='data',
                symmetric=False,
                array=model_df['ci_upper_error'],
                arrayminus=model_df['ci_lower_error']
            )
        )
        
        # Add random baseline as a red dotted horizontal line
        fig.add_hline(
            y=random_baseline_f1,
            line_dash="dot",
            line_color="red",
            line_width=2,
            annotation_text=f"Random Baseline (F1={random_baseline_f1:.3f})",
            annotation_position="right"
        )
        
        # Adjust y-axis range to ensure baseline is visible
        y_min = min(0, random_baseline_f1 - 0.1)
        y_max = max(1, model_df['frequency_weighted_f1'].max() + 0.1)
        
        fig.update_layout(
            yaxis_range=[y_min, y_max],
            xaxis_title="Experiment Configuration",
            yaxis_title=f"Mean Frequency-Weighted F1 Score ({int(CONFIDENCE_LEVEL*100)}% CI)",
            xaxis={'tickangle': 45},
            height=600,
            legend_title="Non-Activating Source"
        )
        
        fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
        fig.show()
        
        # Save the chart with model name in filename
        model_safe = model_name.replace('-', '').replace(' ', '_')
        output_file_pdf = visualizations_dir / f"contrastive_freq_weighted_f1_bar_{model_safe}_mean.pdf"
        output_file_png = visualizations_dir / f"contrastive_freq_weighted_f1_bar_{model_safe}_mean.png"
        fig.write_image(str(output_file_pdf))
        fig.write_image(str(output_file_png))
        print(f"Saved contrastive F1 bar chart for {model_name}: {output_file_pdf}")

print(f"\nBar charts saved to {visualizations_dir}")

### 3.1 Model Comparison Charts

Create side-by-side comparison charts to compare the same configurations across different models.

In [ ]:
# Create comparison charts showing the same configuration across different models
print("\nGenerating model comparison chart (mean of detection + fuzz)...")

if not mean_error_table.empty:
    # Create a grouped bar chart comparing models for each source+ranking combination
    # Pivot the data to have models as separate bars
    comparison_df = mean_error_table.copy()
    comparison_df['config_key'] = comparison_df['source'] + ' (' + comparison_df['ranking'] + ')'
    
    fig = px.bar(
        comparison_df,
        x='config_key',
        y='frequency_weighted_f1',
        color='model',
        barmode='group',
        title=f'Model Comparison - Mean Frequency-Weighted F1 Score (Detection + Fuzz) ({int(CONFIDENCE_LEVEL*100)}% CI)',
        text='frequency_weighted_f1',
        hover_data=['experiment', 'mode', 'n_score_types']
    )
    
    # Add error bars
    fig.update_traces(
        error_y=dict(
            type='data',
            symmetric=False,
            array=comparison_df['ci_upper_error'],
            arrayminus=comparison_df['ci_lower_error']
        )
    )
    
    fig.update_layout(
        yaxis_range=[0, 1],
        xaxis_title="Configuration (Source + Ranking)",
        yaxis_title=f"Mean Frequency-Weighted F1 Score ({int(CONFIDENCE_LEVEL*100)}% CI)",
        xaxis={'tickangle': 45},
        height=600,
        legend_title="Model"
    )
    
    fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
    fig.show()
    
    # Save the comparison chart
    output_file_pdf = visualizations_dir / f"contrastive_model_comparison_mean.pdf"
    output_file_png = visualizations_dir / f"contrastive_model_comparison_mean.png"
    fig.write_image(str(output_file_pdf))
    fig.write_image(str(output_file_png))
    print(f"Saved model comparison chart: {output_file_pdf}")
else:
    print("No data available for model comparison chart")

print(f"\nModel comparison chart saved to {visualizations_dir}")

## 4. Generate KDE Distribution Plots

Create kernel density estimation plots showing the distribution of F1 scores across different experimental configurations.

In [ ]:
def prepare_kde_data_mean(experiment_results, score_types=['detection', 'fuzz']):
    """Prepare data for KDE plots by extracting per-latent mean F1 scores across score types."""
    kde_data = []
    
    for exp_name, data in experiment_results.items():
        latent_df = data['latent_df']
        counts = data['counts']
        config = data['config']
        
        if counts is None:
            continue
        
        # Collect F1 scores for each latent across score types
        latent_f1_map = {}  # (module, latent_idx) -> list of f1 scores
        
        for score_type in score_types:
            score_subset = latent_df[latent_df['score_type'] == score_type]
            
            if len(score_subset) == 0:
                continue
            
            # Extract per-latent F1 scores for this score type
            for (module, latent_idx), grp in score_subset.groupby(["module", "latent_idx"]):
                if module in counts and latent_idx < len(counts[module]):
                    f1 = compute_classification_metrics(compute_confusion(grp))["f1_score"]
                    firing_count = counts[module][latent_idx].item()
                    
                    if (module, latent_idx) not in latent_f1_map:
                        latent_f1_map[(module, latent_idx)] = {
                            'f1_scores': [],
                            'firing_count': firing_count
                        }
                    latent_f1_map[(module, latent_idx)]['f1_scores'].append(float(f1))
        
        # Compute mean F1 for each latent
        for (module, latent_idx), info in latent_f1_map.items():
            if len(info['f1_scores']) > 0:
                mean_f1 = np.mean(info['f1_scores'])
                kde_data.append({
                    'experiment': exp_name,
                    'source': config['source'],
                    'ranking': config['ranking'],
                    'mode': config['mode'],
                    'f1_score': mean_f1,
                    'firing_count': info['firing_count'],
                    'module': module,
                    'latent_idx': latent_idx,
                    'n_score_types': len(info['f1_scores'])
                })
    
    return pd.DataFrame(kde_data)

# Set up matplotlib style for KDE plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Generate KDE plots using mean of detection and fuzz
print(f"Generating KDE plots for mean F1 (detection + fuzz)...")

kde_data = prepare_kde_data_mean(experiment_results, score_types=['detection', 'fuzz'])

if kde_data.empty:
    print(f"No data available for mean F1 KDE plot")
else:
    # Create separate plots for each ranking strategy
    ranking_strategies = ['quantiles', 'top']
    
    for ranking_strategy in ranking_strategies:
        ranking_data = kde_data[kde_data['ranking'] == ranking_strategy]
        
        if ranking_data.empty:
            print(f"No data for {ranking_strategy} strategy in mean F1")
            continue
        
        # Create figure comparing all sources for this ranking strategy
        fig, ax = plt.subplots(1, 1, figsize=(12, 6))
        
        sources = sorted(ranking_data['source'].unique())
        
        for source in sources:
            source_data = ranking_data[ranking_data['source'] == source]
            
            if len(source_data) > 0:
                # Weight by firing count for the KDE
                weights = source_data['firing_count'].values
                weights = weights / weights.sum()  # Normalize weights
                
                # Create KDE plot with clean labels
                source_label = SOURCE_DISPLAY.get(source, source)
                sns.kdeplot(
                    data=source_data,
                    x='f1_score',
                    weights=weights,
                    ax=ax,
                    label=source_label,
                    color=SOURCE_COLORS.get(source, '#808080'),
                    linewidth=3,
                    alpha=0.8
                )
        
        # Styling
        ranking_display = RANKING_DISPLAY.get(ranking_strategy, ranking_strategy)
        ax.set_title(f'Mean F1 Score Distribution (Detection + Fuzz) - {ranking_display} Strategy', 
                     fontsize=16, fontweight='bold', pad=20)
        ax.set_xlabel('Mean F1 Score', fontsize=14, fontweight='medium')
        ax.set_ylabel('Density', fontsize=14, fontweight='medium')
        ax.legend(title='Non-Activating Source', title_fontsize=12, fontsize=11, frameon=True, 
                 fancybox=True, shadow=True, framealpha=0.9)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0, 1)
        
        # Add subtle background styling
        ax.set_facecolor('#FAFAFA')
        
        plt.tight_layout()
        
        # Show the plot
        plt.show()
        
        # Save the plot
        output_file_pdf = visualizations_dir / f"contrastive_kde_mean_{ranking_strategy}.pdf"
        output_file_png = visualizations_dir / f"contrastive_kde_mean_{ranking_strategy}.png"
        plt.savefig(str(output_file_pdf), dpi=300, bbox_inches='tight', facecolor='white')
        plt.savefig(str(output_file_png), dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Saved KDE plot: {output_file_pdf}")
        
        plt.close()

print(f"\nKDE plots saved to {visualizations_dir}")

## 6. Random vs Co-occurrence Performance Comparison

Scatter plot analysis to see if the performance drop from random to co-occurrence non-activating sources is consistent across latents, or if some latents drop more than others.

In [ ]:
def prepare_random_vs_cooccurrence_comparison_mean(experiment_results, score_types=['detection', 'fuzz']):
    """Prepare data comparing random vs co-occurrence mean FREQUENCY-WEIGHTED performance per latent.
    
    Note: This computes the mean of frequency-weighted F1 scores across score types (detection, fuzz).
    Each latent's F1 is weighted by its firing frequency within each score type, then averaged.
    """
    comparison_data = []
    
    # Group experiments by model and ranking to find matching pairs
    for model_name in set(v['config']['model_display'] for v in experiment_results.values()):
        for ranking_strategy in ['top', 'quantiles']:
            # Find random and co-occurrence experiments for this model/ranking combo
            random_exp = None
            cooccurrence_exp = None
            
            for exp_name, data in experiment_results.items():
                config = data['config']
                if (config['model_display'] == model_name and 
                    config['ranking'] == ranking_strategy):
                    if config['source'] == 'random':
                        random_exp = exp_name
                    elif config['source'] == 'co-occurrence':
                        cooccurrence_exp = exp_name
            
            # If we have both, compare them
            if random_exp and cooccurrence_exp:
                random_data = experiment_results[random_exp]
                cooccur_data = experiment_results[cooccurrence_exp]
                
                random_latent_df = random_data['latent_df']
                cooccur_latent_df = cooccur_data['latent_df']
                counts = random_data['counts']
                
                if counts is None:
                    continue
                
                # Compute frequency-weighted F1 for each score type, then average
                random_freq_weighted_f1_by_score = {}
                cooccur_freq_weighted_f1_by_score = {}
                
                for score_type in score_types:
                    # Random: compute frequency-weighted F1
                    random_subset = random_latent_df[random_latent_df['score_type'] == score_type]
                    if len(random_subset) > 0:
                        random_freq_weighted_f1_by_score[score_type] = frequency_weighted_f1(random_subset, counts)
                    
                    # Co-occurrence: compute frequency-weighted F1
                    cooccur_subset = cooccur_latent_df[cooccur_latent_df['score_type'] == score_type]
                    if len(cooccur_subset) > 0:
                        cooccur_freq_weighted_f1_by_score[score_type] = frequency_weighted_f1(cooccur_subset, counts)
                
                # Compute mean frequency-weighted F1 across score types
                if len(random_freq_weighted_f1_by_score) > 0 and len(cooccur_freq_weighted_f1_by_score) > 0:
                    random_mean_freq_f1 = np.mean(list(random_freq_weighted_f1_by_score.values()))
                    cooccur_mean_freq_f1 = np.mean(list(cooccur_freq_weighted_f1_by_score.values()))
                    
                    # Get total firing count for this model/ranking (sum across all latents)
                    total_firing_count = sum(counts[module].sum().item() for module in counts.keys())
                    
                    comparison_data.append({
                        'model': model_name,
                        'ranking': ranking_strategy,
                        'random_f1': random_mean_freq_f1,
                        'cooccurrence_f1': cooccur_mean_freq_f1,
                        'f1_drop': random_mean_freq_f1 - cooccur_mean_freq_f1,
                        'firing_count': total_firing_count,
                        'n_score_types': len(random_freq_weighted_f1_by_score)
                    })
    
    return pd.DataFrame(comparison_data)

# Generate comparison table using frequency-weighted mean F1 scores
print(f"Generating random vs co-occurrence comparison for frequency-weighted mean F1 (detection + fuzz)...")

comparison_df = prepare_random_vs_cooccurrence_comparison_mean(experiment_results, score_types=['detection', 'fuzz'])

if comparison_df.empty:
    print(f"No comparison data available for frequency-weighted mean F1")
else:
    print("\nFrequency-Weighted F1 Comparison (Random vs Co-occurrence):")
    print("=" * 80)
    print(comparison_df.to_string(index=False))
    print("=" * 80)
    
    # Create a simple bar chart comparison
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    
    # Prepare data for grouped bar chart
    x = np.arange(len(comparison_df))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, comparison_df['random_f1'], width, 
                   label='Random', color=SOURCE_COLORS['random'], alpha=0.8)
    bars2 = ax.bar(x + width/2, comparison_df['cooccurrence_f1'], width,
                   label='Co-occurrence', color=SOURCE_COLORS['co-occurrence'], alpha=0.8)
    
    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}',
                   ha='center', va='bottom', fontsize=9)
    
    # Styling
    ax.set_xlabel('Model and Ranking Strategy', fontsize=12, fontweight='medium')
    ax.set_ylabel('Frequency-Weighted Mean F1 Score', fontsize=12, fontweight='medium')
    ax.set_title('Random vs Co-occurrence: Frequency-Weighted Mean F1 Comparison\n(Mean of Detection + Fuzz)',
                fontsize=14, fontweight='bold', pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{row['model']}\n{row['ranking']}" for _, row in comparison_df.iterrows()], 
                       rotation=0, ha='center')
    ax.legend(fontsize=11, frameon=True, fancybox=True, shadow=True)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 1)
    ax.set_facecolor('#FAFAFA')
    
    plt.tight_layout()
    plt.show()
    
    # Save the plot
    output_file_pdf = visualizations_dir / f"random_vs_cooccurrence_comparison_mean.pdf"
    output_file_png = visualizations_dir / f"random_vs_cooccurrence_comparison_mean.png"
    plt.savefig(str(output_file_pdf), dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(str(output_file_png), dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\nSaved comparison plot: {output_file_pdf}")
    
    plt.close()

print(f"\nComparison saved to {visualizations_dir}")